Шаг. 1

In [ ]:
!pip install spacy
!python -m spacy download ru_core_news_sm

  Using cached ru_core_news_sm-3.8.0-py3-none-any.whl (15.3 MB)
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import math
import spacy
import nltk
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')
nlp = spacy.load('ru_core_news_sm')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Шаг. 2

In [ ]:
corpus = [
    "Сегодня была отличная погода, и мы пошли гулять в красивый парк!",
    "Ужасная погода ... Весь день идет сильный дождь, прогулка отменяется.",
    "В парке мы видели красивых птиц. Погода шепчет, солнце светит ярко.",
    "Дождь не прекращается! Сильный ветер сорвал все планы на вечер."
]

sample_text = corpus[0]
print("Исходный текст для анализа:", sample_text)

Исходный текст для анализа: Сегодня была отличная погода, и мы пошли гулять в красивый парк!


Шаг. 3

In [ ]:
text_lowercase = sample_text.lower()

text_cleaned = re.sub(r'[^а-яё\s]', '', text_lowercase)

print("Текст после очистки RegEx:")
print(text_cleaned)

Текст после очистки RegEx:
сегодня была отличная погода и мы пошли гулять в красивый парк


Шаг. 4

In [ ]:
russian_stopwords = set(stopwords.words('russian'))

doc = nlp(text_cleaned)

filtered_tokens = []
for token in doc:
  lemma = token.lemma_
  if lemma not in russian_stopwords and lemma.strip():
    filtered_tokens.append(lemma)

print("Токены после лемматизации и удаления стоп-слов:")
print(filtered_tokens)

Токены после лемматизации и удаления стоп-слов:
['сегодня', 'отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']


Шаг. 5

In [ ]:
def pipeline_preprocess(text):
  text = text.lower()
  text = re.sub(r'[^а-яё\s]', '', text)
  doc = nlp(text)
  return [token.lemma_ for token in doc if token.lemma_ not in russian_stopwords and token.lemma_.strip()]

processed_corpus = [pipeline_preprocess(text) for text in corpus]

for i, doc_tokens in enumerate(processed_corpus):
  print(f"Документ {i+1}: {doc_tokens}")

Документ 1: ['сегодня', 'отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']
Документ 2: ['ужасный', 'погода', 'весь', 'день', 'идти', 'сильный', 'дождь', 'прогулка', 'отменяться']
Документ 3: ['парк', 'видеть', 'красивый', 'птица', 'погода', 'шепчет', 'солнце', 'светить', 'ярко']
Документ 4: ['дождь', 'прекращаться', 'сильный', 'ветер', 'сорвать', 'весь', 'план', 'вечер']


Шаг. 6

In [ ]:
vocabulary = sorted(list(set([word for doc in processed_corpus for word in doc])))
print("Уникальный словарь корпуса (всего слов:", len(vocabulary), "):")
print(vocabulary)

def calculate_tfidf(word, current_doc, all_docs):
  tf = current_doc.count(word) / len(current_doc) if len(current_doc) > 0 else 0

  docs_with_word = sum(1 for d in all_docs if word in d)
  idf = math.log(len(all_docs) / docs_with_word)

  return round(tf * idf, 4)

print("\nВекторы документов (TF-IDF числовые представления):")
for i, doc in enumerate(processed_corpus):
  vector = []
  for word in vocabulary:
    tfidf_value = calculate_tfidf(word, doc, processed_corpus)
    vector.append(tfidf_value)
  print(f"Вектор Документа {i+1}: {vector}")

Уникальный словарь корпуса (всего слов: 26 ):
['весь', 'ветер', 'вечер', 'видеть', 'гулять', 'день', 'дождь', 'идти', 'красивый', 'отличный', 'отменяться', 'парк', 'план', 'погода', 'пойти', 'прекращаться', 'прогулка', 'птица', 'светить', 'сегодня', 'сильный', 'солнце', 'сорвать', 'ужасный', 'шепчет', 'ярко']

Векторы документов (TF-IDF числовые представления):
Вектор Документа 1: [0.0, 0.0, 0.0, 0.0, 0.198, 0.0, 0.0, 0.0, 0.099, 0.198, 0.0, 0.099, 0.0, 0.0411, 0.198, 0.0, 0.0, 0.0, 0.0, 0.198, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Вектор Документа 2: [0.077, 0.0, 0.0, 0.0, 0.0, 0.154, 0.077, 0.154, 0.0, 0.0, 0.154, 0.0, 0.0, 0.032, 0.0, 0.0, 0.154, 0.0, 0.0, 0.0, 0.077, 0.0, 0.0, 0.154, 0.0, 0.0]
Вектор Документа 3: [0.0, 0.0, 0.0, 0.154, 0.0, 0.0, 0.0, 0.0, 0.077, 0.0, 0.0, 0.077, 0.0, 0.032, 0.0, 0.0, 0.0, 0.154, 0.154, 0.0, 0.0, 0.154, 0.0, 0.0, 0.154, 0.154]
Вектор Документа 4: [0.0866, 0.1733, 0.1733, 0.0, 0.0, 0.0, 0.0866, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1733, 0.0, 0.0, 0.1733, 0.0, 0.0, 0.0

Задание 1.1

In [ ]:
print("Количество стоп-слов:", len(russian_stopwords))
russian_stopwords_sort=sorted(russian_stopwords)
print("Список стоп-слов в алфавитном порядке:\n", russian_stopwords_sort)

Количество стоп-слов: 151
Список стоп-слов в алфавитном порядке:
 ['а', 'без', 'более', 'больше', 'будет', 'будто', 'бы', 'был', 'была', 'были', 'было', 'быть', 'в', 'вам', 'вас', 'вдруг', 'ведь', 'во', 'вот', 'впрочем', 'все', 'всегда', 'всего', 'всех', 'всю', 'вы', 'где', 'да', 'даже', 'два', 'для', 'до', 'другой', 'его', 'ее', 'ей', 'ему', 'если', 'есть', 'еще', 'ж', 'же', 'за', 'зачем', 'здесь', 'и', 'из', 'или', 'им', 'иногда', 'их', 'к', 'как', 'какая', 'какой', 'когда', 'конечно', 'кто', 'куда', 'ли', 'лучше', 'между', 'меня', 'мне', 'много', 'может', 'можно', 'мой', 'моя', 'мы', 'на', 'над', 'надо', 'наконец', 'нас', 'не', 'него', 'нее', 'ней', 'нельзя', 'нет', 'ни', 'нибудь', 'никогда', 'ним', 'них', 'ничего', 'но', 'ну', 'о', 'об', 'один', 'он', 'она', 'они', 'опять', 'от', 'перед', 'по', 'под', 'после', 'потом', 'потому', 'почти', 'при', 'про', 'раз', 'разве', 'с', 'сам', 'свою', 'себе', 'себя', 'сейчас', 'со', 'совсем', 'так', 'такой', 'там', 'тебя', 'тем', 'теперь', 'то', 

Задание 1.3

In [ ]:
russian_stopwords.update(['сегодня','весь'])

processed_corpus = [pipeline_preprocess(text) for text in corpus]

print("Документ 1:",processed_corpus[0])

Документ 1: ['отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']


Задание 2.1

In [ ]:
corpus.append("За окном сгущается гроза. Погода явно не для прогулки!")

Задание 2.2

In [ ]:
processed_corpus = [pipeline_preprocess(text) for text in corpus]

for i, doc_tokens in enumerate(processed_corpus):
  print(f"Документ {i+1}: {doc_tokens}")

Документ 1: ['отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']
Документ 2: ['ужасный', 'погода', 'день', 'идти', 'сильный', 'дождь', 'прогулка', 'отменяться']
Документ 3: ['парк', 'видеть', 'красивый', 'птица', 'погода', 'шепчет', 'солнце', 'светить', 'ярко']
Документ 4: ['дождь', 'прекращаться', 'сильный', 'ветер', 'сорвать', 'план', 'вечер']
Документ 5: ['окно', 'сгущаться', 'гроза', 'погода', 'явно', 'прогулка']


In [ ]:
vocabulary = sorted(list(set([word for doc in processed_corpus for word in doc])))
print("Уникальный словарь корпуса (всего слов:", len(vocabulary), "):")
print(vocabulary)

print("\nВекторы документов (TF-IDF числовые представления):")
for i, doc in enumerate(processed_corpus):
  vector = []
  for word in vocabulary:
    tfidf_value = calculate_tfidf(word, doc, processed_corpus)
    vector.append(tfidf_value)
  print(f"Вектор Документа {i+1}: {vector}")

Уникальный словарь корпуса (всего слов: 28 ):
['ветер', 'вечер', 'видеть', 'гроза', 'гулять', 'день', 'дождь', 'идти', 'красивый', 'окно', 'отличный', 'отменяться', 'парк', 'план', 'погода', 'пойти', 'прекращаться', 'прогулка', 'птица', 'светить', 'сгущаться', 'сильный', 'солнце', 'сорвать', 'ужасный', 'шепчет', 'явно', 'ярко']

Векторы документов (TF-IDF числовые представления):
Вектор Документа 1: [0.0, 0.0, 0.0, 0.0, 0.2682, 0.0, 0.0, 0.0, 0.1527, 0.0, 0.2682, 0.0, 0.1527, 0.0, 0.0372, 0.2682, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Вектор Документа 2: [0.0, 0.0, 0.0, 0.0, 0.0, 0.2012, 0.1145, 0.2012, 0.0, 0.0, 0.0, 0.2012, 0.0, 0.0, 0.0279, 0.0, 0.0, 0.1145, 0.0, 0.0, 0.0, 0.1145, 0.0, 0.0, 0.2012, 0.0, 0.0, 0.0]
Вектор Документа 3: [0.0, 0.0, 0.1788, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1018, 0.0, 0.0, 0.0, 0.1018, 0.0, 0.0248, 0.0, 0.0, 0.0, 0.1788, 0.1788, 0.0, 0.0, 0.1788, 0.0, 0.0, 0.1788, 0.0, 0.1788]
Вектор Документа 4: [0.2299, 0.2299, 0.0, 0.0, 0.0, 0.0, 0.1309, 

Задание 3.1 и 3.2

In [ ]:
corpus[0]="13 дней назад была отличная погода, и мы пошли гулять в красивый парк!"

def pipeline_preprocess(text):
  text = text.lower()
  text = re.sub(r'[^а-яё\s\d]', '', text)
  doc = nlp(text)
  return [token.lemma_ for token in doc if token.lemma_ not in russian_stopwords and token.lemma_.strip()]

processed_corpus = [pipeline_preprocess(text) for text in corpus]

print("Документ 1:",processed_corpus[0])

Документ 1: ['13', 'день', 'назад', 'отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']
